# H&M 고객 데이터 해설/정답

In [2]:
import pandas as pd

df = pd.read_csv("customer_hm.csv")
print(df.shape)
df.head()

(1048575, 6)


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0,ACTIVE,NONE,49
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0,0,ACTIVE,NONE,25
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0,0,ACTIVE,NONE,24
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0,0,ACTIVE,NONE,54
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1,1,ACTIVE,Regularly,52


### 문제 1 해설 — 그룹화: `club_member_status`별 고객 수

`size()`는 **행 개수**를 그대로 세어줍니다. (결측 포함/제외 이슈도 적고, 카운트용으로 딱 좋음)

- 포인트: `groupby(...).size()`
- 정렬: `.sort_values(ascending=False)`

In [3]:
# groupby + size (가장 깔끔한 카운트)
status_cnt = df.groupby("club_member_status").size().sort_values(ascending=False)
status_cnt

club_member_status
ACTIVE        982635
PRE-CREATE     65581
LEFT CLUB        359
dtype: int64

### 문제 2 해설 — 집계표(agg): 상태별 고객 수 + 비율(%)

`agg`로 **카운트 컬럼을 먼저 만든 뒤**, `.assign()`으로 비율을 붙이면 흐름이 깔끔합니다.

- `count=('customer_id','size')` : 어떤 열을 지정해도 되고, `size`가 핵심
- 비율: 전체 합으로 나누고 *100

In [9]:
status_summary = (
    df.groupby("club_member_status")
      .agg(count=("customer_id", "size"))
      .assign(ratio_pct=lambda x: x["count"] / x["count"].sum() * 100)
      .sort_values("count", ascending=False)
)
status_summary


,count,ratio_pct
club_member_status,,
ACTIVE,982635,93.711466
PRE-CREATE,65581,6.254297
LEFT CLUB,359,0.034237


In [10]:
status_summary = (
    df.groupby("club_member_status")
      .agg(count=("customer_id", "size"))
      .sort_values("count", ascending=False)
)
status_summary['ratio_pct'] = status_summary['count'] / status_summary['count'].sum() * 100
status_summary

,count,ratio_pct
club_member_status,,
ACTIVE,982635,93.711466
PRE-CREATE,65581,6.254297
LEFT CLUB,359,0.034237


### 문제 3 해설 — 그룹화 + 다중 집계: `fashion_news_frequency`별 나이 요약

`agg`는 여러 통계를 **한 번에** 뽑을 때 최고입니다.

- 평균/중앙값 같이 보기: 데이터가 치우쳤는지 감 잡기 좋음
- `.round(2)`로 보기 좋게

In [11]:
fn_age = (
    df.groupby("fashion_news_frequency")
      .agg(
          count=("customer_id", "size"),
          age_mean=("age", "mean"),
          age_median=("age", "median"),
      )
      .assign(age_mean=lambda x: x["age_mean"].round(2))
      .sort_values("count", ascending=False)
)
fn_age

,count,age_mean,age_median
fashion_news_frequency,,,
NONE,674698,36.00,31.0
Regularly,373218,37.03,32.0
Monthly,658,39.39,36.5


In [12]:
fn_age = (
    df.groupby("fashion_news_frequency")
      .agg(
          count=("customer_id", "size"),
          age_mean=("age", "mean"),
          age_median=("age", "median"),
      )
      .sort_values("count", ascending=False)
)
fn_age['age_mean'] = fn_age['age_mean'].round(2)
fn_age

,count,age_mean,age_median
fashion_news_frequency,,,
NONE,674698,36.00,31.0
Regularly,373218,37.03,32.0
Monthly,658,39.39,36.5


### 문제 4 해설 — 복수 기준 그룹화: `club_member_status × Active` 요약

0/1 데이터의 평균은 **1의 비율**로 해석할 수 있음. (`FN`이 0/1이라서 가능)

- MultiIndex 결과가 나오면 다음 문제(unstack/reshape)에서 맛있게 씁니다 ^^ 꺼억

In [17]:
status_active = (
    df.groupby(["club_member_status", "Active"])
      .agg(
          count=("customer_id", "size"),
          age_mean=("age", "mean"),
          fn_rate=("FN", "mean"),
      )
)
status_active


count   age_mean   fn_rate
club_member_status Active                             
ACTIVE             0       624068  35.537385  0.015623
                   1       358567  37.003241  1.000000
LEFT CLUB          0          356  33.750000  0.002809
                   1            3  44.333333  1.000000
PRE-CREATE         0        61205  40.458296  0.001846
                   1         4376  46.056673  1.000000

### 문제 5 해설 — transform: 상태별 평균 나이를 원본 길이 유지로 붙이기

`transform`의 장점: **집계 결과를 원본 행 수 그대로 돌려준다** -> 바로 새 컬럼으로 붙이기 좋음.

- `groupby(...).mean()`은 길이가 줄어든 요약표
- `groupby(...).transform('mean')`은 원본 길이 유지

In [18]:
df["age_mean_by_status"] = df.groupby("club_member_status")["age"].transform("mean")
df["age_diff"] = df["age"] - df["age_mean_by_status"]

df[["club_member_status", "age", "age_mean_by_status", "age_diff"]].head()


,club_member_status,age,age_mean_by_status,age_diff
0,ACTIVE,49,36.072281,12.927719
1,ACTIVE,25,36.072281,-11.072281
2,ACTIVE,24,36.072281,-12.072281
3,ACTIVE,54,36.072281,17.927719
4,ACTIVE,52,36.072281,15.927719


### 문제 6 해설 — transform(응용): (상태, 뉴스빈도) 그룹 크기 붙이기

`transform('size')`는 그룹별 행 개수를 원본 길이로 되돌려줍니다.

- `drop_duplicates()`로 조합별 대표 행만 남기면 TOP 뽑기 쉬움

In [23]:
df["group_size"] = df.groupby(["club_member_status", "fashion_news_frequency"])["customer_id"].transform("size")

top5_groups = (
    df[["club_member_status", "fashion_news_frequency", "group_size"]]
      .drop_duplicates()
      .sort_values("group_size", ascending=False)
      .head(5)
)
top5_groups


,club_member_status,fashion_news_frequency,group_size
0,ACTIVE,NONE,613304.0
4,ACTIVE,Regularly,368719.0
33,PRE-CREATE,NONE,61039.0
24,PRE-CREATE,Regularly,4495.0
4911,ACTIVE,Monthly,611.0


### 문제 7 해설 — unstack: 상태 × 뉴스빈도 고객 수 표 만들기

`groupby` 결과가 MultiIndex Series로 나오면, `unstack()`으로 **보고서형 표**가 됩니다.

- 행: club_member_status
- 열: fashion_news_frequency
- 값: 고객 수

In [30]:
status_fn_table = (
    df.groupby(["club_member_status", "fashion_news_frequency"])["customer_id"]
      .size()
      .unstack(fill_value=0)
)
status_fn_table


fashion_news_frequency,Monthly,NONE,Regularly
club_member_status,,,
ACTIVE,611,613304,368719
LEFT CLUB,0,355,4
PRE-CREATE,47,61039,4495


### 문제 8 해설 — pivot_table: 상태 × 뉴스빈도 활동률(Active 평균)

`pivot_table`은 엑셀 피벗처럼 **집계까지 포함해서** wide 표를 만들어줍니다.

- `pivot`은 중복 조합이 있으면 에러가 날 수 있지만
- `pivot_table`은 `aggfunc`로 중복을 처리합니다.

In [29]:
active_pivot = df.pivot_table(
    values="Active",
    index="club_member_status",
    columns="fashion_news_frequency",
    aggfunc="mean",
    fill_value=0
)
active_pivot


fashion_news_frequency,Monthly,NONE,Regularly
club_member_status,,,
ACTIVE,0.944354,0.000623,0.969866
LEFT CLUB,0.000000,0.000000,0.750000
PRE-CREATE,0.978723,0.000082,0.962180


### 문제 9 해설 — melt: 8번 피벗 표를 long(그래프용)으로 바꾸기

피벗 표는 보기엔 좋은데, 그래프/시각화용으로는 long이 더 편합니다.

- `reset_index()`로 인덱스를 컬럼으로 내려준 다음
- `melt()`로 길게 펼치기

In [31]:
active_long = (
    active_pivot
      .reset_index()
      .melt(id_vars="club_member_status",
            var_name="fashion_news_frequency",
            value_name="active_rate")
)
active_long.head()


,club_member_status,fashion_news_frequency,active_rate
0,ACTIVE,Monthly,0.944354
1,LEFT CLUB,Monthly,0.000000
2,PRE-CREATE,Monthly,0.978723
3,ACTIVE,NONE,0.000623
4,LEFT CLUB,NONE,0.000000


### 문제 10 해설 — pivot: 9번 long 데이터를 다시 wide로 복원

`pivot`은 (index, columns) 조합이 **유일할 때** 딱 깔끔하게 복원됩니다.

- `equals`로 값/인덱스/컬럼까지 동일한지 확인 가능

In [32]:
active_wide_back = active_long.pivot(
    index="club_member_status",
    columns="fashion_news_frequency",
    values="active_rate"
)

print(active_wide_back.equals(active_pivot))
active_wide_back


True


fashion_news_frequency,Monthly,NONE,Regularly
club_member_status,,,
ACTIVE,0.944354,0.000623,0.969866
LEFT CLUB,0.000000,0.000000,0.750000
PRE-CREATE,0.978723,0.000082,0.962180


### 문제 11 해설 — 문자열 정리: `fashion_news_frequency` 클린 버전 만들기

문자열 컬럼은 **대소문자/공백/결측** 때문에 그룹화가 깨지는 경우가 많습니다.

- `.fillna('unknown')`로 결측을 먼저 처리
- `.str.strip().str.lower()`로 표준화

In [36]:
before = df["fashion_news_frequency"].value_counts(dropna=False)

df["fn_clean"] = (
    df["fashion_news_frequency"]
      .fillna("unknown")
      .astype(str)
      .str.strip()
      .str.lower()
)

after = df["fn_clean"].value_counts(dropna=False)

print("before")
print(before.head(10))
print("\nafter")
print(after.head(10))


before
fashion_news_frequency
NONE         674698
Regularly    373218
Monthly         658
NaN               1
Name: count, dtype: int64

after
fn_clean
none         674698
regularly    373218
monthly         658
unknown           1
Name: count, dtype: int64


### 문제 12 해설 — 문자열: `customer_id` prefix/suffix 추출

문자열 슬라이싱은 `.str[:n]`, `.str[-n:]` 형태로 깔끔하게 됩니다.

- prefix로 간단한 군집/분포 확인할 때 종종 써요.

In [ ]:
df["id_prefix2"] = df["customer_id"].str[:2]
df["id_suffix3"] = df["customer_id"].str[-3:]

df["id_prefix2"].value_counts().head(10)

id_prefix2
b5    5403
83    5398
13    5397
a1    5379
bd    5377
67    5373
60    5360
48    5350
40    5350
57    5348
Name: count, dtype: int64

### 문제 13 해설 — map: `club_member_status` 한글 라벨 붙이기

`map`은 값 치환/라벨링에 제일 간단합니다.

- 매핑에 없는 값이 생기면 NaN이 되므로, `.fillna('미상')` 같이 안전장치 추가

In [47]:
status_map = {
    "ACTIVE": "활동중",
    "PRE-CREATE": "사전생성",
    "LEFT CLUB": "탈퇴",
}

df["status_ko"] = df["club_member_status"].map(status_map).fillna("미상")
df["status_ko"].value_counts()


status_ko
활동중     982635
사전생성     65581
탈퇴         359
Name: count, dtype: int64

### 문제 14 해설 — apply/lambda: 나이대(`age_band`) 구간화 + 활동률 보기

오늘은 apply 연습 겸 구간화를 했지만, 실무에서는 `pd.cut`/`pd.qcut`도 자주 씁니다.

- 0/1 평균 = 비율 (활동률)
- 구간화 후 groupby 하면 패턴이 잘 보입니다.

In [48]:
def to_age_band(x):
    if x < 20:
        return "10대"
    elif x < 30:
        return "20대"
    elif x < 40:
        return "30대"
    elif x < 60:
        return "40~50대"
    else:
        return "60대+"

df["age_band"] = df["age"].apply(to_age_band)

age_band_summary = (
    df.groupby("age_band")
      .agg(count=("customer_id", "size"),
           active_rate=("Active", "mean"))
      .sort_index()
)
age_band_summary


,count,active_rate
age_band,,
10대,55256,0.359581
20대,409196,0.333906
30대,181112,0.313298
40~50대,332160,0.361910
60대+,70851,0.416226


### 문제 15 해설 — set_index / loc: 고객 ID로 빠르게 조회하기

인덱스를 키로 두면 `loc` 조회가 정말 편해집니다.

- `set_index('customer_id')` -> `loc[id]`가 자연스러워짐
- 분석 끝나면 `reset_index()`로 원복하는 습관도 중요

In [56]:
df_idx = df.set_index("customer_id")

ids = df["customer_id"].sample(5, random_state=42).tolist()
picked = df_idx.loc[ids, ["age", "status_ko", "fn_clean"]]
display(picked)

# 원복(인덱스를 다시 컬럼으로)
df = df_idx.reset_index()
df.head()


,age,status_ko,fn_clean
customer_id,,,
95a27f54cded740b1f8a71ffc5a10630c788781abae474d640d247563300fb6c,49,활동중,regularly
b3553fce166fb434ce7862b98cbed186adf77458ee2ec60d5ece2ec559167d57,64,활동중,regularly
ad9e46403030a92dffc30299e1711f9fe791c8ba3a954c28d29defc6025eec8e,43,활동중,none
9626d0cadb8d86f21fead29f2a8c68d8c175004db8def7c7f13b3d8db18dbac9,35,활동중,regularly
7ec984843e492a4f4fb82e981142d48d7d5741e0e0173c93029c55b731427500,30,활동중,regularly


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,age_mean_by_status,age_diff,group_size,fn_clean,id_prefix2,id_suffix3,age_band,status_ko
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0,ACTIVE,NONE,49,36.072281,12.927719,613304.0,none,00,657,40~50대,활동중
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0,0,ACTIVE,NONE,25,36.072281,-11.072281,613304.0,none,00,0fa,20대,활동중
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0,0,ACTIVE,NONE,24,36.072281,-12.072281,613304.0,none,00,318,20대,활동중
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0,0,ACTIVE,NONE,54,36.072281,17.927719,613304.0,none,00,43e,40~50대,활동중
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1,1,ACTIVE,Regularly,52,36.072281,15.927719,368719.0,regularly,00,85a,40~50대,활동중


### 문제 16 해설 — reset_index(정리): 표 결과를 '컬럼형'으로 만들기

피벗/언스택 결과는 보통 인덱스로 뭔가가 올라가요.

- 엑셀로 넘기거나 리포트용으로 쓰려면 `reset_index()`로 **컬럼형**으로 바꿔두는 게 편합니다.

In [57]:
status_fn_table_reset = status_fn_table.reset_index()
status_fn_table_reset.head()

fashion_news_frequency,club_member_status,Monthly,NONE,Regularly
0,ACTIVE,611,613304,368719
1,LEFT CLUB,0,355,4
2,PRE-CREATE,47,61039,4495


### 문제 17 해설 — merge: 뉴스 빈도 점수표 붙이기

`map`으로도 붙일 수 있지만, 이번에는 **merge 감각**을 유지하려고 일부러 매핑 DF를 만들었어요.

- 실무에서 매핑 테이블이 외부 파일/DB로 오는 경우가 많아서 merge를 많이 씁니다.
- `how='left'` : 원본 df 행을 절대 잃지 않기

In [ ]:
fn_score_df = pd.DataFrame({
    "fn_clean": ["none", "monthly", "regularly", "unknown"],
    "fn_score": [0, 1, 2, -1]
})

df2 = df.merge(fn_score_df, on="fn_clean", how="left")

df2.groupby("fn_score")["age"].mean().sort_index()

fn_score
-1    38.000000
 0    36.000507
 1    39.389058
 2    37.030374
Name: age, dtype: float64

### 문제 18 해설 — concat + 최종 reshape: `age_band × status_ko` 활동률 표 만들기

`concat`은 파일이 쪼개져 들어오거나, 조건별로 나눴던 걸 다시 합칠 때 자주 씁니다.

마무리로는 `pivot_table` -> `melt`까지 한 번에 돌려보면서
- wide(보고서용)
- long(그래프용)
두 형태를 모두 다뤄보는 게 목표입니다.

In [62]:
# 분할 -> concat 복원
df_on = df[df["Active"] == 1]
df_off = df[df["Active"] == 0]

df_back = pd.concat([df_on, df_off], axis=0).sort_index()
print("원본 행 수:", len(df), "| 복원 행 수:", len(df_back))

# age_band x status_ko 활동률 표
report_wide = df.pivot_table(
    values="Active",
    index="age_band",
    columns="status_ko",
    aggfunc="mean",
    fill_value=0
)

report_long = (
    report_wide.reset_index()
              .melt(id_vars="age_band", var_name="status_ko", value_name="active_rate")
)

report_wide, report_long


원본 행 수: 1048575 | 복원 행 수: 1048575


(status_ko      사전생성        탈퇴       활동중
 age_band                               
 10대        0.014640  0.000000  0.365437
 20대        0.035436  0.000000  0.347037
 30대        0.059387  0.029851  0.334348
 40~50대     0.079743  0.011364  0.388175
 60대+       0.122456  0.000000  0.441809,
    age_band status_ko  active_rate
 0       10대      사전생성     0.014640
 1       20대      사전생성     0.035436
 2       30대      사전생성     0.059387
 3    40~50대      사전생성     0.079743
 4      60대+      사전생성     0.122456
 5       10대        탈퇴     0.000000
 6       20대        탈퇴     0.000000
 7       30대        탈퇴     0.029851
 8    40~50대        탈퇴     0.011364
 9      60대+        탈퇴     0.000000
 10      10대       활동중     0.365437
 11      20대       활동중     0.347037
 12      30대       활동중     0.334348
 13   40~50대       활동중     0.388175
 14     60대+       활동중     0.441809)